In [302]:
%pip install duckdb


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [303]:
import duckdb
import pandas as pd

con = duckdb.connect("../data/shelterflow.duckdb")
intakes = con.execute("SELECT * FROM bronze_intakes").df()
outcomes = con.execute("SELECT * FROM bronze_outcomes").df()

# Exploratory Data Analysis of Shelter Intakes & Outcomes

## Intakes
### Column Info

In [304]:
intakes.info() 

<class 'pandas.DataFrame'>
RangeIndex: 80187 entries, 0 to 80186
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   age_upon_intake   80187 non-null  str  
 1   animal_id         80187 non-null  str  
 2   animal_type       80187 non-null  str  
 3   breed             80187 non-null  str  
 4   color             80187 non-null  str  
 5   datetime          80187 non-null  str  
 6   datetime2         80187 non-null  str  
 7   found_location    80187 non-null  str  
 8   intake_condition  80187 non-null  str  
 9   intake_type       80187 non-null  str  
 10  name              55603 non-null  str  
 11  sex_upon_intake   80186 non-null  str  
dtypes: str(12)
memory usage: 7.3 MB


### Descriptive Statistics

In [305]:
intakes.describe()

,age_upon_intake,animal_id,animal_type,breed,color,datetime,datetime2,found_location,intake_condition,intake_type,name,sex_upon_intake
count,80187,80187,80187,80187,80187,80187,80187,80187,80187,80187,55603,80186
unique,46,72365,5,2166,529,57055,57055,36772,8,5,14731,5
top,1 year,A721033,Dog,Domestic Shorthair Mix,Black/White,2016-09-23T12:00:00.000,2016-09-23T12:00:00.000,Austin (TX),Normal,Stray,Bella,Intact Male
freq,14672,13,45743,23519,8340,64,64,14443,70520,56280,357,25488


### Missing Values

In [306]:
intakes.isnull().sum()

age_upon_intake         0
animal_id               0
animal_type             0
breed                   0
color                   0
datetime                0
datetime2               0
found_location          0
intake_condition        0
intake_type             0
name                24584
sex_upon_intake         1
dtype: int64

### Duplicate Rows

In [307]:
intakes.duplicated().sum()

26

### <a id="intakes-breeds"></a>Breed Distribution

In [308]:
intakes["breed"].value_counts()

breed
Domestic Shorthair Mix              23519
Pit Bull Mix                         6382
Chihuahua Shorthair Mix              4860
Labrador Retriever Mix               4841
Domestic Medium Hair Mix             2337
                                    ...  
Siamese/Snowshoe                        1
Yorkshire Terrier/Dachshund             1
Guinea                                  1
Norfolk Terrier/Pug                     1
Australian Shepherd/Basset Hound        1
Name: count, Length: 2166, dtype: int64

### Intake Type Distribution

In [309]:
intakes["intake_type"].value_counts()

intake_type
Stray                 56280
Owner Surrender       15156
Public Assist          5033
Wildlife               3467
Euthanasia Request      251
Name: count, dtype: int64

### Animal Type Distribution

In [310]:
intakes["animal_type"].value_counts()

animal_type
Dog          45743
Cat          29659
Other         4434
Bird           342
Livestock        9
Name: count, dtype: int64

### Top 50 Dog Breeds

In [311]:
intakes[intakes["animal_type"] == "Dog"]["breed"].value_counts().head(50)

breed
Pit Bull Mix                          6382
Chihuahua Shorthair Mix               4860
Labrador Retriever Mix                4841
German Shepherd Mix                   1963
Australian Cattle Dog Mix             1105
Dachshund Mix                          813
Boxer Mix                              689
Border Collie Mix                      667
Miniature Poodle Mix                   662
Catahoula Mix                          484
Australian Shepherd Mix                470
Rat Terrier Mix                        469
Yorkshire Terrier Mix                  449
Siberian Husky Mix                     442
Jack Russell Terrier Mix               437
Miniature Schnauzer Mix                404
Beagle Mix                             394
Staffordshire Mix                      385
Chihuahua Longhair Mix                 371
Great Pyrenees Mix                     350
Cairn Terrier Mix                      350
Pointer Mix                            338
Rottweiler Mix                         327
Ameri

### Count of Unique Dog Breeds

In [312]:
intakes[intakes["animal_type"] == "Dog"]["breed"].nunique()

1925

In [313]:
pd.set_option("display.max_rows", None)

### Pit Bull Breed Variations

In [314]:
intakes[intakes["breed"].str.contains("pit bull", case=False, na=False)]["breed"].value_counts()

breed
Pit Bull Mix                                           6382
Pit Bull                                                234
Labrador Retriever/Pit Bull                             218
American Pit Bull Terrier Mix                           182
Pit Bull/Labrador Retriever                             107
Pit Bull/Boxer                                           53
Boxer/Pit Bull                                           49
Pit Bull/Australian Cattle Dog                           29
Pit Bull/Catahoula                                       28
American Bulldog/Pit Bull                                19
Pit Bull/Chinese Sharpei                                 18
Australian Cattle Dog/Pit Bull                           18
Pit Bull/Pointer                                         18
German Shepherd/Pit Bull                                 16
Pit Bull/Chihuahua Shorthair                             12
Pit Bull/German Shepherd                                 11
Pointer/Pit Bull                  

#### Verify that "Pit Bull" is not written as "Pitbull" anywhere in the dataset:

In [315]:
intakes[intakes["breed"].str.contains("pitbull", case=False, na=False)]["breed"].value_counts()

Series([], Name: count, dtype: int64)

### Labrador Retriever Breed Variations

In [316]:
intakes[intakes["breed"].str.contains("labrador", case=False, na=False)]["breed"].value_counts()

breed
Labrador Retriever Mix                               4841
Labrador Retriever                                    241
Labrador Retriever/Pit Bull                           218
German Shepherd/Labrador Retriever                    160
Labrador Retriever/German Shepherd                    140
Pit Bull/Labrador Retriever                           107
Labrador Retriever/Border Collie                       80
Labrador Retriever/Australian Cattle Dog               74
Border Collie/Labrador Retriever                       71
Australian Cattle Dog/Labrador Retriever               62
Labrador Retriever/Boxer                               52
Labrador Retriever/Great Pyrenees                      51
Labrador Retriever/Beagle                              45
Labrador Retriever/Pointer                             43
Labrador Retriever/Chow Chow                           39
Boxer/Labrador Retriever                               38
Pointer/Labrador Retriever                             37
Labrador

#### Verify that "Labrador Retriever" is not abbreviated as "Lab" anywhere in the dataset:

In [317]:
intakes[intakes["breed"].str.contains("^lab ", case=False, na=False)]["breed"].value_counts()

Series([], Name: count, dtype: int64)

### Chihuahua Breed Variations

In [318]:
intakes[intakes["breed"].str.contains("chihuahua", case=False, na=False)]["breed"].value_counts()

breed
Chihuahua Shorthair Mix                            4860
Chihuahua Longhair Mix                              371
Chihuahua Shorthair/Dachshund                       252
Chihuahua Shorthair                                 204
Dachshund/Chihuahua Shorthair                       179
Chihuahua Shorthair/Rat Terrier                      61
Chihuahua Shorthair/Jack Russell Terrier             59
Chihuahua Shorthair/Pug                              51
Rat Terrier/Chihuahua Shorthair                      49
Jack Russell Terrier/Chihuahua Shorthair             41
Pug/Chihuahua Shorthair                              37
Chihuahua Shorthair/Cardigan Welsh Corgi             30
Miniature Pinscher/Chihuahua Shorthair               25
Cairn Terrier/Chihuahua Shorthair                    23
Beagle/Chihuahua Shorthair                           23
Yorkshire Terrier/Chihuahua Shorthair                23
Chihuahua Shorthair/Beagle                           21
Chihuahua Shorthair/Yorkshire Terrier     

### Dachshund Breed Variations

In [319]:
intakes[intakes["breed"].str.contains("dachshund", case=False, na=False)]["breed"].value_counts()

breed
Dachshund Mix                               813
Chihuahua Shorthair/Dachshund               252
Dachshund/Chihuahua Shorthair               179
Dachshund                                   102
Dachshund Longhair Mix                       95
Dachshund Wirehair Mix                       87
Dachshund/Beagle                             24
Beagle/Dachshund                             18
Dachshund Longhair                           17
Dachshund/Rat Terrier                        15
Dachshund/Labrador Retriever                 15
Labrador Retriever/Dachshund                 15
Dachshund Wirehair/Chihuahua Shorthair       13
Jack Russell Terrier/Dachshund               12
Dachshund Wirehair/Miniature Poodle          11
Dachshund/Manchester Terrier                 10
Dachshund Wirehair/Border Terrier             8
Chihuahua Shorthair/Dachshund Wirehair        8
Miniature Poodle/Dachshund                    8
Dachshund/Pit Bull                            8
Manchester Terrier/Dachshund      

### German Shepherd Breed Variations

In [320]:
intakes[intakes["breed"].str.contains("german shepherd", case=False, na=False)]["breed"].value_counts()

breed
German Shepherd Mix                                   1963
German Shepherd                                        236
German Shepherd/Labrador Retriever                     160
Labrador Retriever/German Shepherd                     140
Siberian Husky/German Shepherd                          33
German Shepherd/Chow Chow                               28
Border Collie/German Shepherd                           28
Australian Cattle Dog/German Shepherd                   23
German Shepherd/Siberian Husky                          22
German Shepherd/Rottweiler                              21
German Shepherd/Australian Cattle Dog                   20
Beagle/German Shepherd                                  19
German Shepherd/Pit Bull                                16
German Shepherd/Boxer                                   15
Chow Chow/German Shepherd                               15
Rottweiler/German Shepherd                              14
Pit Bull/German Shepherd                          

### Husky Breed Variations

In [321]:
intakes[intakes["breed"].str.contains("husky", case=False, na=False)]["breed"].value_counts()

breed
Siberian Husky Mix                          442
Siberian Husky                              115
Alaskan Husky Mix                            57
Siberian Husky/German Shepherd               33
German Shepherd/Siberian Husky               22
Labrador Retriever/Siberian Husky            20
Siberian Husky/Labrador Retriever            15
Siberian Husky/Australian Cattle Dog         13
German Shepherd/Alaskan Husky                10
Alaskan Husky                                 9
Siberian Husky/Border Collie                  6
Labrador Retriever/Alaskan Husky              6
Siberian Husky/Pit Bull                       4
Alaskan Husky/German Shepherd                 4
Rottweiler/Siberian Husky                     4
Pit Bull/Alaskan Husky                        4
Akita/Siberian Husky                          4
Alaskan Husky/Australian Shepherd             3
Siberian Husky/Alaskan Malamute               3
Chow Chow/Siberian Husky                      2
Australian Cattle Dog/Siberian Hus

### Australian Cattle Dog Breed Variations

In [322]:
intakes[intakes["breed"].str.contains("australian cattle", case=False, na=False)]["breed"].value_counts()

breed
Australian Cattle Dog Mix                               1105
Australian Cattle Dog                                     87
Labrador Retriever/Australian Cattle Dog                  74
Australian Cattle Dog/Labrador Retriever                  62
Pit Bull/Australian Cattle Dog                            29
Australian Cattle Dog/German Shepherd                     23
German Shepherd/Australian Cattle Dog                     20
Australian Cattle Dog/Pit Bull                            18
Australian Cattle Dog/Border Collie                       17
Border Collie/Australian Cattle Dog                       15
Australian Cattle Dog/Boxer                               13
Siberian Husky/Australian Cattle Dog                      13
Jack Russell Terrier/Australian Cattle Dog                13
Australian Cattle Dog/Beagle                              11
Australian Cattle Dog/Pointer                             10
Cardigan Welsh Corgi/Australian Cattle Dog                 9
Catahoula/Australi

### Staffordshire Breed Variations

In [323]:
intakes[intakes["breed"].str.contains("staffordshire", case=False, na=False)]["breed"].value_counts()

breed
Staffordshire Mix                                       385
American Staffordshire Terrier Mix                      219
American Staffordshire Terrier                           26
Staffordshire                                            17
Boxer/Staffordshire                                      10
American Staffordshire Terrier/Labrador Retriever         7
Australian Cattle Dog/Staffordshire                       7
Staffordshire/Labrador Retriever                          5
Labrador Retriever/Staffordshire                          5
Staffordshire/Boxer                                       4
Staffordshire/Beagle                                      3
American Staffordshire Terrier/American Bulldog           3
American Staffordshire Terrier/Pit Bull                   3
Boxer/American Staffordshire Terrier                      3
Staffordshire/English Bulldog                             2
American Staffordshire Terrier/Boxer                      2
Staffordshire/Blue Lacy           

### Top 50 Cat Breeds

In [324]:
intakes[intakes["animal_type"] == "Cat"]["breed"].value_counts().head(50)

breed
Domestic Shorthair Mix             23519
Domestic Medium Hair Mix            2337
Domestic Longhair Mix               1254
Siamese Mix                          999
Domestic Shorthair                   387
American Shorthair Mix               213
Snowshoe Mix                         146
Domestic Medium Hair                 133
Maine Coon Mix                       104
Manx Mix                              80
Russian Blue Mix                      63
Siamese                               63
Domestic Longhair                     44
Himalayan Mix                         34
Ragdoll Mix                           23
Persian Mix                           19
Siamese/Domestic Shorthair            13
Bengal Mix                            13
Angora Mix                            12
Balinese Mix                          11
American Curl Shorthair Mix           11
Japanese Bobtail Mix                  10
Maine Coon                            10
Persian                                9
Tonkinese 

### Count of Unique Cat Breeds

In [325]:
intakes[intakes["animal_type"] == "Cat"]["breed"].nunique()

83

### Domestic Shorthair Breed Variations

In [326]:
intakes[intakes["breed"].str.contains("domestic shorthair", case=False, na=False)]["breed"].value_counts()

breed
Domestic Shorthair Mix                   23519
Domestic Shorthair                         387
Siamese/Domestic Shorthair                  13
Manx/Domestic Shorthair                      4
Bengal/Domestic Shorthair                    2
Domestic Shorthair/Siamese                   2
Domestic Shorthair/Manx                      1
Domestic Shorthair/Maine Coon                1
Snowshoe/Domestic Shorthair                  1
Domestic Shorthair/Abyssinian                1
Domestic Shorthair/British Shorthair         1
Scottish Fold/Domestic Shorthair             1
Domestic Shorthair/Domestic Shorthair        1
Name: count, dtype: int64

#### Verify that "Shorthair" is not written as "Short Hair" anywhere in the dataset:

In [327]:
intakes[intakes["breed"].str.contains("domestic short hair", case=False, na=False)]["breed"].value_counts()

Series([], Name: count, dtype: int64)

### Domestic Longhair Breed Variations

In [328]:
intakes[intakes["breed"].str.contains("domestic longhair", case=False, na=False)]["breed"].value_counts()

breed
Domestic Longhair Mix                  1254
Domestic Longhair                        44
Domestic Longhair/Persian                 2
Domestic Longhair/Rex                     2
Manx/Domestic Longhair                    2
Domestic Longhair/Domestic Longhair       1
Domestic Longhair/Russian Blue            1
Name: count, dtype: int64

#### Verify that "Longhair" is not written as "Long Hair" anywhere in the dataset:

In [329]:
intakes[intakes["breed"].str.contains("domestic long hair", case=False, na=False)]["breed"].value_counts()

Series([], Name: count, dtype: int64)

### Domestic Medium Hair Breed Variations

In [330]:
intakes[intakes["breed"].str.contains("domestic medium hair", case=False, na=False)]["breed"].value_counts()

breed
Domestic Medium Hair Mix           2337
Domestic Medium Hair                133
Domestic Medium Hair/Siamese          3
Domestic Medium Hair/Maine Coon       2
Domestic Medium Hair/Manx             1
Name: count, dtype: int64

#### Verify that "Medium Hair" is not written as "Mediumhair" anywhere in the dataset:

In [331]:
intakes[intakes["breed"].str.contains("domestic mediumhair", case=False, na=False)]["breed"].value_counts()

Series([], Name: count, dtype: int64)

### Investigating Breed Name Abbreviations

#### Verifying that there are no other breed name abbreviations aside from "Oriental Sh":

In [332]:
intakes[intakes["breed"].str.contains(" sh ", case=False, na=False)]["breed"].value_counts()

breed
Rabbit Sh Mix      294
Oriental Sh Mix      3
Name: count, dtype: int64

#### <a id="oriental-sh-mix"></a>Verifying that there are only "Oriental Sh Mix" instances:

In [333]:
intakes[intakes["breed"].str.contains("oriental", case=False, na=False)]["breed"].value_counts()

breed
Oriental Sh Mix    3
Name: count, dtype: int64

In [334]:
pd.reset_option("display.max_rows")

### <a id="datetime"></a>Verifying the Similarity of the `datetime` and `datetime2` columns

#### Checking if every row is identical:

In [335]:
(intakes["datetime"] == intakes["datetime2"]).all()

True

#### Counting the differences in `datetime` and `datetime2` values for each row

In [336]:
(intakes["datetime"] != intakes["datetime2"]).sum()

0

### Profiling the `age_upon_intake` column

In [337]:
intakes["age_upon_intake"].value_counts().head(30)

age_upon_intake
1 year       14672
2 years      11617
1 month       7468
3 years       5250
2 months      4027
4 years       3102
4 weeks       2827
5 years       2755
3 weeks       2286
4 months      2049
5 months      2009
3 months      1938
6 years       1869
2 weeks       1599
7 years       1596
6 months      1546
8 years       1542
9 months      1197
10 years      1174
7 months      1156
8 months       903
9 years        878
10 months      714
1 week         668
12 years       595
1 weeks        528
11 months      527
0 years        456
11 years       454
13 years       380
Name: count, dtype: int64

In [338]:
intakes[intakes["age_upon_intake"].str.contains("days|day", case=False, na=False)]["age_upon_intake"].value_counts()

age_upon_intake
3 days    375
1 day     349
2 days    271
4 days    199
6 days    190
5 days    125
Name: count, dtype: int64


## Outcomes
### Column Info

In [339]:
outcomes.info()

<class 'pandas.DataFrame'>
RangeIndex: 80681 entries, 0 to 80680
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   age_upon_outcome  80673 non-null  str  
 1   animal_id         80681 non-null  str  
 2   animal_type       80681 non-null  str  
 3   breed             80681 non-null  str  
 4   color             80681 non-null  str  
 5   date_of_birth     80681 non-null  str  
 6   datetime          80681 non-null  str  
 7   monthyear         80681 non-null  str  
 8   name              56116 non-null  str  
 9   outcome_subtype   36893 non-null  str  
 10  outcome_type      80667 non-null  str  
 11  sex_upon_outcome  80679 non-null  str  
dtypes: str(12)
memory usage: 7.4 MB


### Descriptive Statistics

In [340]:
outcomes.describe()

,age_upon_outcome,animal_id,animal_type,breed,color,date_of_birth,datetime,monthyear,name,outcome_subtype,outcome_type,sex_upon_outcome
count,80673,80681,80681,80681,80681,80681,80681,80681,56116,36893,80667,80679
unique,46,72877,5,2176,532,5956,66474,66474,14824,19,9,5
top,1 year,A721033,Dog,Domestic Shorthair Mix,Black/White,2014-05-05T00:00:00,2016-04-18T00:00:00,2016-04-18T00:00:00,Bella,Partner,Adoption,Neutered Male
freq,14911,13,45856,23821,8396,112,39,39,362,20082,34232,28702


### Missing Values

In [341]:
outcomes.isnull().sum()

age_upon_outcome        8
animal_id               0
animal_type             0
breed                   0
color                   0
date_of_birth           0
datetime                0
monthyear               0
name                24565
outcome_subtype     43788
outcome_type           14
sex_upon_outcome        2
dtype: int64

### Duplicate Rows

In [342]:
outcomes.duplicated().sum()

10

### <a id="outcomes-breeds"></a>Breed Distribution

In [343]:
outcomes["breed"].value_counts()

breed
Domestic Shorthair Mix          23821
Pit Bull Mix                     6361
Chihuahua Shorthair Mix          4874
Labrador Retriever Mix           4849
Domestic Medium Hair Mix         2368
                                ...  
Guinea                              1
Norfolk Terrier/Pug                 1
Pomeranian/Beagle                   1
Chickadee Mix                       1
Labrador Retriever/Pekingese        1
Name: count, Length: 2176, dtype: int64

### Outcome Type Distribution

In [344]:
outcomes["outcome_type"].value_counts()

outcome_type
Adoption           34232
Transfer           24050
Return to Owner    14851
Euthanasia          6289
Died                 699
Disposal             304
Rto-Adopt            179
Missing               47
Relocate              16
Name: count, dtype: int64

### Animal Type Distribution

In [345]:
outcomes["animal_type"].value_counts()

animal_type
Dog          45856
Cat          30028
Other         4446
Bird           341
Livestock       10
Name: count, dtype: int64

### Profiling the `age_upon_outcome` column

In [346]:
outcomes["age_upon_outcome"].value_counts().head(30)

age_upon_outcome
1 year       14911
2 years      11683
2 months      9377
3 years       5326
3 months      3481
1 month       3424
4 years       3092
5 years       2789
4 months      2470
5 months      1995
6 months      1940
6 years       1878
8 years       1608
7 years       1588
3 weeks       1474
2 weeks       1348
10 months     1241
8 months      1221
4 weeks       1206
10 years      1205
7 months      1001
9 years        855
9 months       718
12 years       619
1 weeks        528
11 months      520
11 years       447
1 week         429
13 years       403
14 years       260
Name: count, dtype: int64

In [347]:
outcomes[outcomes["age_upon_outcome"].str.contains("days|day", case=False, na=False)]["age_upon_outcome"].value_counts()

age_upon_outcome
3 days    239
2 days    226
1 day     157
6 days    153
4 days    140
5 days    116
Name: count, dtype: int64

### Profiling the `outcome_type` column

In [348]:
outcomes["outcome_type"].value_counts()

outcome_type
Adoption           34232
Transfer           24050
Return to Owner    14851
Euthanasia          6289
Died                 699
Disposal             304
Rto-Adopt            179
Missing               47
Relocate              16
Name: count, dtype: int64

### <a id="outcome-subtypes"></a>Profiling the `outcome_subtype` column

In [349]:
outcomes["outcome_subtype"].value_counts()

outcome_subtype
Partner                20082
Foster                  5714
SCRP                    3211
Suffering               2563
Rabies Risk             2546
Snr                      755
Aggressive               508
Offsite                  367
In Kennel                354
Medical                  268
In Foster                183
Behavior                 142
At Vet                    71
Enroute                   49
Underage                  28
Court/Investigation       23
In Surgery                17
Possible Theft             9
Barn                       3
Name: count, dtype: int64

## <a id="intakes-vs-outcomes"></a>Comparing Intakes & Outcomes

### Breeds in the Outcomes Dataset That Are Not in the Intakes Dataset

In [350]:
outcomes[~outcomes["breed"].isin(intakes["breed"])]["breed"].value_counts()

breed
Cold Water Mix                                      2
Manchester Terrier                                  1
Chinese Sharpei/Bull Terrier                        1
Beagle/Border Terrier                               1
Bull Terrier/Border Collie                          1
Border Terrier/Miniature Schnauzer                  1
Pointer/Chinese Sharpei                             1
Domestic Shorthair/Domestic Medium Hair             1
Akita/Chow Chow                                     1
Pit Bull/Border Terrier                             1
Hovawart Mix                                        1
Dogue De Bordeaux/Labrador Retriever                1
Miniature                                           1
Doberman Pinsch/Catahoula                           1
Nova Scotia Duck Tolling Retriever/Border Collie    1
Chickadee Mix                                       1
Labrador Retriever/Pekingese                        1
Name: count, dtype: int64

### Breeds in the Intakes Dataset That Are Not in the Outcomes Dataset

In [351]:
intakes[~intakes["breed"].isin(outcomes["breed"])]["breed"].value_counts()

breed
Chihuahua Longhair/Silky Terrier          1
German Shorthair Pointer/Border Collie    1
Doberman Pinsch/Greyhound                 1
Toy Poodle/Chihuahua Shorthair            1
German Shepherd/Bloodhound                1
Pointer/Beagle                            1
Siamese/Snowshoe                          1
Name: count, dtype: int64


## Key Findings

### Common Values

* The most common dog breed at the shelter is a Pit Bull Mix (seen in both [intakes](#intakes-breeds) and [outcomes](#outcomes-breeds)).
* The most common cat breed at the shelter is a Domestic Shorthair Mix (seen in both [intakes](#intakes-breeds) and [outcomes](#outcomes-breeds)).
    * It is worth noting that this is a very generalized breed which does not take into account cat color (which prospective adopters may factor into their decision to adopt). Thus, animal breed and color may need to be considered together for accurate analysis.
* The majority of animals at the shelter are dogs, followed by cats.
  
### What to Fix

* Standardize `breed` values for cats and dogs. Upon comparing `breed` values for `intakes` vs. `outcomes`, I concluded that the same standardization rules apply to both datasets ([see here](#intakes-vs-outcomes)).
    * Standardize `breed` values for cross-breeds as `Mix` (e.g. `Pug/Chihuahua` becomes `Pug Mix`).
    * Standardize `breed` value `American Pit Bull Terrier` to `Pit Bull`.
    * Standardize `breed` value `Queensland Heeler` to `Australian Cattle Dog`. These are the same dog breed, with the more common naming being `Australian Cattle Dog` ([see here](#Top-50-Dog-Breeds)).
    * Standardize `breed` value `Oriental Sh Mix` to `Oriental Shorthair Mix`.
        * I confirmed that there are only instances of `Oriental Sh Mix`, no purebreds ([see here](#oriental-sh-mix)).
* `datetime`, `datetime2`, `date_of_birth`, and `monthyear` are stored as `str` type instead of `datetime` type.
    * `datetime` will be converted 
* There is a large amount of null `name` values that need to be handled.
* `age_upon_intake` and `age_upon_outcome` are currently stored as `str` type and require parsing.
  
### Other Notes

* There are similarly named `breed` values `Staffordshire` and `American Staffordshire Terrier` ([see here](#Staffordshire-Breed-Variations)). I left the ambiguous `Staffordshire` breed entries unchanged because I couldn't confirm the intended breed without additional context (`Staffordshire` could refer to the `Staffordshire Bull Terrier` or the `American Staffordshire Terrier`, two distinct breeds).
* Upon profiling the `outcome_subtype` column ([see here](#outcome-subtypes)), I noted a few abbreviations, namely `Snr` (most likely meaning `Shelter-Neuter-Release`) and `SCRP` (most likely meaning `Stray Cat Return Program`). However, I left these ambiguous entries unchanged because I could not confirm their meaning without additional context. This is not of concern, as `outcome_subtype` will not be heavily used during gold layer analytics, which will rely moreso on the generalized `outcome_type`.